# M6 W6 v2 retrieval evaluation

This notebook evaluates the provisional three-book, 5,976-chunk v2 handoff
with the shared project retriever. It does not contain a second BM25,
dense-search, or RRF implementation.

Protocol:

- Development: BM25 plus nine dense models, each with Dense and six Hybrid
  weights (64 configurations).
- Candidate depth: project default `rrf_input_top_k=20`.
- Baseline decision: replace BM25 only when the best Dev candidate improves
  complete Hit@5, does not reduce MRR, and passes an exact paired McNemar test
  at `p < 0.05`.
- Diagnostic Test comparison: BM25 and BGE-M3 Hybrid
  (Dense 0.10 / BM25 0.90). These 26 questions were previously evaluated across
  all configurations, so the results are diagnostic and are not a pristine
  one-time Test or a basis for model selection.
- The 162/26 split is provisional; the team handoff specifies a future formal
  six-textbook 60/180 Dev/Test evaluation.
- All result tables displayed below are loaded from generated CSV files.


## 1. Configuration and reproducibility

Inputs must be supplied through `CS30_M6_*` environment variables or placed in
`eval_inputs/m6_v2/`. There is deliberately no Desktop fallback. Large indexes
remain under ignored `artifacts/`; the small manifest and summary CSVs are written
to `docs/results/m6_w6_v2/` for review.


In [ ]:
from __future__ import annotations

import collections
import gc
import hashlib
import json
import math
import os
import re
import sys
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "cs30").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from the cs30-personalised-rag checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cs30.config import AppConfig, RetrievalConfig  # noqa: E402
from cs30.contracts import RetrievalMode  # noqa: E402
from cs30.evaluation.manifest import capture_git_state  # noqa: E402
from cs30.pipeline import build_real_retrieval_deps  # noqa: E402
from cs30.retrieval.v2_adapter import materialise_v2_retrieval_view  # noqa: E402

INPUT_DIR = Path(
    os.getenv("CS30_M6_INPUT_DIR", PROJECT_ROOT / "eval_inputs" / "m6_v2")
).expanduser()


def resolve_input(env_name: str, filename: str) -> Path:
    configured = os.getenv(env_name)
    candidate = Path(configured).expanduser() if configured else INPUT_DIR / filename
    if not candidate.is_file():
        raise FileNotFoundError(
            f"Missing {filename}. Set {env_name} or place it under {INPUT_DIR}."
        )
    return candidate.resolve()


CORPUS_ZIP = resolve_input(
    "CS30_M6_CORPUS_ZIP",
    "v2_structure_aware_corpus_3book_dev_main_3553825.zip",
)
M5_ZIP = resolve_input(
    "CS30_M6_INDEX_ZIP",
    "M5_v2_candidate_indexes_2026-10-09.zip",
)
INTEGRATED_GOLD = resolve_input(
    "CS30_M6_GOLD",
    "gold_v0_1_1_integrated.jsonl",
)

WORK_DIR = Path(
    os.getenv(
        "CS30_M6_WORK_DIR",
        PROJECT_ROOT / "artifacts" / "m6_w6_retrieval",
    )
).expanduser()
RESULT_DIR = Path(
    os.getenv(
        "CS30_M6_RESULT_DIR",
        PROJECT_ROOT / "docs" / "results" / "m6_w6_v2",
    )
).expanduser()
RUN_EXPENSIVE = os.getenv("CS30_M6_RUN_MODELS", "0") == "1"
DEVICE = os.getenv("CS30_M6_DEVICE", "cuda")

TOP_K = (1, 3, 5, 10)
FINAL_TOP_K = max(TOP_K)
RRF_K = 60
RRF_INPUT_TOP_K = 20
HYBRID_WEIGHTS = (
    (0.95, 0.05),
    (0.90, 0.10),
    (0.80, 0.20),
    (0.75, 0.25),
    (0.50, 0.50),
    (0.25, 0.75),
)  # (BM25, Dense)
DIAGNOSTIC_BGE_M3 = {
    "model": "bge_m3",
    "mode": "hybrid_d0.10_b0.90",
    "dense_weight": 0.10,
    "bm25_weight": 0.90,
    "selection_status": "diagnostic_not_dev_selected",
}

print("Repository:", PROJECT_ROOT)
print("Inputs:", INPUT_DIR)
print("Results:", RESULT_DIR)
print("Expensive run:", RUN_EXPENSIVE, "device:", DEVICE)

## 2. Model registry, pinned revisions, and input validation


In [ ]:
MODEL_SPECS = {
    "minilm": {
        "model": "sentence-transformers/all-MiniLM-L6-v2",
        "revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
        "dimension": 384,
        "batch": 64,
    },
    "mpnet": {
        "model": "sentence-transformers/all-mpnet-base-v2",
        "revision": "e8c3b32edf5434bc2275fc9bab85f82640a19130",
        "dimension": 768,
        "batch": 32,
    },
    "e5_base": {
        "model": "intfloat/e5-base-v2",
        "revision": "f52bf8ec8c7124536f0efb74aca902b2995e5bcd",
        "dimension": 768,
        "batch": 32,
        "query_instruction": "query: ",
        "document_prefix": "passage: ",
    },
    "bge_base": {
        "model": "BAAI/bge-base-en-v1.5",
        "revision": "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a",
        "dimension": 768,
        "batch": 32,
        "query_instruction": "Represent this sentence for searching relevant passages: ",
    },
    "jina": {
        "model": "jinaai/jina-embeddings-v2-small-en",
        "revision": "44e7d1d6caec8c883c2d4b207588504d519788d0",
        "dimension": 512,
        "batch": 16,
        "trust_remote_code": True,
    },
    "gte_modernbert": {
        "model": "Alibaba-NLP/gte-modernbert-base",
        "revision": "e7f32e3c00f91d699e8c43b53106206bcc72bb22",
        "dimension": 768,
        "batch": 4,
        "trust_remote_code": True,
        "m5_path": "gte-modernbert",
    },
    "nomic": {
        "model": "nomic-ai/nomic-embed-text-v1.5",
        "revision": "e9b6763023c676ca8431644204f50c2b100d9aab",
        "dimension": 768,
        "batch": 8,
        "trust_remote_code": True,
        "query_instruction": "search_query: ",
        "document_prefix": "search_document: ",
    },
    "bge_m3": {
        "model": "BAAI/bge-m3",
        "revision": "5617a9f61b028005a4858fdac845db406aefb181",
        "dimension": 1024,
        "batch": 4,
        "m5_path": "bge-m3",
    },
    "qwen3": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
        "dimension": 1024,
        "batch": 2,
        "m5_path": "qwen3-embedding",
        "query_prompt_name": "query",
    },
}
for spec in MODEL_SPECS.values():
    spec.setdefault("query_instruction", "")
    spec.setdefault("query_prompt_name", None)
    spec.setdefault("document_prefix", "")
    spec.setdefault("trust_remote_code", False)
    assert re.fullmatch(r"[0-9a-f]{40}", spec["revision"])


def sha256(data: bytes) -> str:
    return "sha256:" + hashlib.sha256(data).hexdigest()


def read_jsonl(data: bytes) -> list[dict]:
    text = data.decode("utf-8-sig")
    return [json.loads(line) for line in text.splitlines() if line.strip()]


with zipfile.ZipFile(CORPUS_ZIP) as archive:
    corpus_bytes = archive.read("records.jsonl")
    corpus_manifest = json.loads(archive.read("manifest.json"))
    chunks = read_jsonl(corpus_bytes)
    for line in archive.read("SHA256SUMS.txt").decode("utf-8").splitlines():
        expected, filename = line.split(None, 1)
        assert hashlib.sha256(archive.read(filename.strip())).hexdigest() == expected

gold_bytes = INTEGRATED_GOLD.read_bytes()
gold = read_jsonl(gold_bytes)
ordered_chunk_ids = [row["chunk_id"] for row in chunks]
by_chunk = {row["chunk_id"]: row for row in chunks}

assert sha256(corpus_bytes) == corpus_manifest["corpus_hash"]
assert len(chunks) == corpus_manifest["record_count"] == 5976
assert len(ordered_chunk_ids) == len(set(ordered_chunk_ids))
assert len(gold) == 188
assert len({row["question_id"] for row in gold}) == 188
assert all(row["answerable"] is True for row in gold)
assert {row["corpus_version"] for row in gold} == {corpus_manifest["corpus_version"]}

WORKING_SPLIT_POLICY = {
    "pending": "dev",
    "proposed_test": "test",
    "dev": "dev",
    "test": "test",
}
dev_gold = [row for row in gold if WORKING_SPLIT_POLICY.get(row["split"]) == "dev"]
test_gold = [row for row in gold if WORKING_SPLIT_POLICY.get(row["split"]) == "test"]
assert len(dev_gold) == 162 and len(test_gold) == 26
assert not ({row["question_id"] for row in dev_gold} & {row["question_id"] for row in test_gold})

print("Models:", len(MODEL_SPECS))
print("Corpus chunks:", len(chunks))
print("Working Dev/Test:", len(dev_gold), len(test_gold))

## 3. Bind Gold evidence spans to v2 chunk IDs


In [ ]:
def normalized(text: str) -> str:
    return " ".join(text.casefold().split())


block_to_chunks = collections.defaultdict(set)
for chunk in chunks:
    for span in chunk["spans"]:
        block_to_chunks[span["block_id"]].add(chunk["chunk_id"])

binding_rows = []
gold_paths = {}
for question in gold:
    paths = []
    for group in question["gold_core_evidence_sets"]:
        mapped = []
        for span in group:
            candidates = block_to_chunks.get(span.get("block_id"), set())
            assert len(candidates) == 1, (question["question_id"], span.get("span_id"))
            chunk_id = next(iter(candidates))
            chunk = by_chunk[chunk_id]
            assert str(chunk["chapter_id"]) == str(span["chapter_id"])
            assert normalized(span["verbatim_text"]) in normalized(chunk["text"])
            mapped.append(chunk_id)
            binding_rows.append(
                {
                    "question_id": question["question_id"],
                    "span_id": span["span_id"],
                    "block_id": span["block_id"],
                    "chunk_id": chunk_id,
                    "textbook_id": chunk["textbook_id"],
                    "method": "unique_block_id+chapter+verbatim",
                }
            )
        paths.append(frozenset(mapped))
    assert paths
    gold_paths[question["question_id"]] = paths

binding_bytes = ("\n".join(json.dumps(row, sort_keys=True) for row in binding_rows) + "\n").encode()
print("Bound Gold spans:", len(binding_rows), "SHA-256:", sha256(binding_bytes))

## 4. Validate M5 candidates and prepare project-retriever views

The adapter validates v2 identity and chunk order, then writes the v1 compatibility
layout consumed by `build_real_retrieval_deps`. It does not implement retrieval.
Locally built indexes use the same pinned revisions and corpus order.


In [ ]:
M5_ARTIFACTS = {}
M5_INDEX_MEMBERS = {}
with zipfile.ZipFile(M5_ZIP) as archive:
    members = {name.replace("\\", "/"): name for name in archive.namelist()}
    assert len(members) == len(archive.namelist())
    for key, spec in MODEL_SPECS.items():
        if "m5_path" not in spec:
            continue
        prefix = f"v2-m5-candidates/{spec['m5_path']}/"
        artifact = json.loads(archive.read(members[prefix + "artifact.json"]))
        ids_bytes = archive.read(members[prefix + "index/chunk_ids.json"])
        index_member = members[prefix + "index/index.faiss"]
        index_bytes = archive.read(index_member)
        assert artifact["corpus_hash"] == corpus_manifest["corpus_hash"]
        assert artifact["manifest_hash"] == corpus_manifest["manifest_hash"]
        assert artifact["chunk_config_hash"] == corpus_manifest["chunk_config_hash"]
        assert artifact["chunk_ids"] == ordered_chunk_ids
        assert json.loads(ids_bytes) == ordered_chunk_ids
        assert sha256(ids_bytes) == artifact["metadata"]["chunk_ids_sha256"]
        assert sha256(index_bytes) == artifact["metadata"]["index_sha256"]
        assert artifact["embedding_model"] == spec["model"]
        assert artifact["embedding_revision"] == spec["revision"]
        assert artifact["embedding_dimension"] == spec["dimension"]
        assert artifact["similarity_metric"] == "inner_product"
        assert artifact["normalise_embeddings"] is True
        M5_ARTIFACTS[key] = artifact
        M5_INDEX_MEMBERS[key] = index_member

display(
    pd.DataFrame(
        [
            {
                "model_key": key,
                "embedding_model": artifact["embedding_model"],
                "revision": artifact["embedding_revision"],
                "dimension": artifact["embedding_dimension"],
            }
            for key, artifact in M5_ARTIFACTS.items()
        ]
    )
)

In [ ]:
def local_v2_artifact(key: str, index_path: Path) -> dict:
    spec = MODEL_SPECS[key]
    return {
        "schema_version": "2.0",
        "artifact_id": f"m6-local-{key}-{corpus_manifest['corpus_hash'][-12:]}",
        "index_type": "faiss.IndexFlatIP",
        "corpus_hash": corpus_manifest["corpus_hash"],
        "chunk_config_hash": corpus_manifest["chunk_config_hash"],
        "chunk_count": len(chunks),
        "chunk_ids": ordered_chunk_ids,
        "embedding_model": spec["model"],
        "embedding_revision": spec["revision"],
        "embedding_dimension": spec["dimension"],
        "similarity_metric": "inner_product",
        "normalise_embeddings": True,
        "index_version": "m6-local-faiss-flat-ip-1",
        "metadata": {"index_sha256": sha256(index_path.read_bytes())},
    }


def ensure_raw_index(key: str) -> tuple[dict, Path]:
    spec = MODEL_SPECS[key]
    raw_dir = WORK_DIR / "raw_indexes" / key
    raw_dir.mkdir(parents=True, exist_ok=True)
    index_path = raw_dir / "index.faiss"
    local_manifest = raw_dir / "build.json"

    if key in M5_ARTIFACTS:
        expected_hash = M5_ARTIFACTS[key]["metadata"]["index_sha256"]
        if not index_path.is_file() or sha256(index_path.read_bytes()) != expected_hash:
            with zipfile.ZipFile(M5_ZIP) as archive:
                index_path.write_bytes(archive.read(M5_INDEX_MEMBERS[key]))
        return M5_ARTIFACTS[key], index_path

    expected = {
        "corpus_hash": corpus_manifest["corpus_hash"],
        "embedding_model": spec["model"],
        "embedding_revision": spec["revision"],
        "dimension": spec["dimension"],
        "document_prefix": spec["document_prefix"],
        "normalise_embeddings": True,
    }
    reusable = False
    if local_manifest.is_file() and index_path.is_file():
        saved = json.loads(local_manifest.read_text(encoding="utf-8"))
        reusable = all(saved.get(name) == value for name, value in expected.items())
        reusable = reusable and saved.get("index_sha256") == sha256(index_path.read_bytes())
    if not reusable:
        import faiss
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(
            spec["model"],
            revision=spec["revision"],
            device=DEVICE,
            trust_remote_code=spec["trust_remote_code"],
        )
        texts = [spec["document_prefix"] + (row.get("embed_text") or row["text"]) for row in chunks]
        vectors = model.encode(
            texts,
            batch_size=spec["batch"],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        ).astype("float32")
        assert vectors.shape == (len(chunks), spec["dimension"])
        index = faiss.IndexFlatIP(spec["dimension"])
        index.add(vectors)
        faiss.write_index(index, str(index_path))
        saved = {
            **expected,
            "index_sha256": sha256(index_path.read_bytes()),
            "built_at_utc": datetime.now(UTC).isoformat(),
        }
        local_manifest.write_text(json.dumps(saved, indent=2) + "\n", encoding="utf-8")
        del model, vectors, index
        gc.collect()
    return local_v2_artifact(key, index_path), index_path


def prepare_retrieval_view(key: str) -> tuple[Path, dict]:
    artifact, index_path = ensure_raw_index(key)
    view_dir = WORK_DIR / "project_views" / key
    spec = MODEL_SPECS[key]
    materialise_v2_retrieval_view(
        v2_artifact=artifact,
        records=chunks,
        index_source=index_path,
        output_dir=view_dir,
        query_instruction=spec["query_instruction"],
        query_prompt_name=spec["query_prompt_name"],
    )
    return view_dir, artifact

## 5. Truncation diagnostics


In [ ]:
def truncation_diagnostic(key: str) -> dict:
    from sentence_transformers import SentenceTransformer

    spec = MODEL_SPECS[key]
    model = SentenceTransformer(
        spec["model"],
        revision=spec["revision"],
        device=DEVICE,
        trust_remote_code=spec["trust_remote_code"],
    )
    max_length = int(model.max_seq_length)
    truncated = 0
    maximum_observed = 0
    batch_size = 128
    texts = [spec["document_prefix"] + (row.get("embed_text") or row["text"]) for row in chunks]
    for start in range(0, len(texts), batch_size):
        encoded = model.tokenizer(
            texts[start : start + batch_size],
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
        lengths = [len(token_ids) for token_ids in encoded]
        maximum_observed = max(maximum_observed, max(lengths, default=0))
        truncated += sum(length > max_length for length in lengths)
    row = {
        "model": key,
        "embedding_model": spec["model"],
        "embedding_revision": spec["revision"],
        "max_input_tokens": max_length,
        "maximum_observed_tokens": maximum_observed,
        "chunk_count": len(texts),
        "truncated_chunk_count": truncated,
        "truncated_chunk_rate": truncated / len(texts),
    }
    del model
    gc.collect()
    return row


if RUN_EXPENSIVE:
    truncation_summary = pd.DataFrame([truncation_diagnostic(key) for key in MODEL_SPECS])
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    truncation_summary.to_csv(RESULT_DIR / "truncation_diagnostics.csv", index=False)
    display(truncation_summary)
else:
    print("Truncation diagnostics run with CS30_M6_RUN_MODELS=1.")

## 6. Shared project retriever

Every measured result below comes from a retriever returned by
`build_real_retrieval_deps`. Project tokenization, query-side stopwords, Dense
normalization, candidate depth, and weighted RRF therefore stay aligned with the
application.


In [ ]:
def build_project_retriever(
    key: str,
    mode: RetrievalMode,
    *,
    dense_weight: float = 0.10,
    bm25_weight: float = 0.90,
):
    view_dir, artifact = prepare_retrieval_view(key)
    spec = MODEL_SPECS[key]
    config = AppConfig(
        fixture_mode=False,
        retrieval=RetrievalConfig(
            mode=mode,
            index_dir=str(view_dir),
            rrf_k=RRF_K,
            rrf_input_top_k=RRF_INPUT_TOP_K,
            rrf_dense_weight=dense_weight,
            rrf_bm25_weight=bm25_weight,
            bm25_stopwords=True,
            expected_embedding_model=spec["model"],
            expected_embedding_revision=spec["revision"],
            embedding_trust_remote_code=spec["trust_remote_code"],
            query_prompt_name=spec["query_prompt_name"],
        ),
    )
    return build_real_retrieval_deps(config).retriever, artifact


def hit_pairs(result) -> list[tuple[str, float]]:
    return [(hit.chunk_id, float(hit.score)) for hit in result.hits]


def candidate_rows(question: dict, model: str, mode: str, result) -> list[dict]:
    rows = []
    for hit in result.hits:
        chunk = by_chunk[hit.chunk_id]
        rows.append(
            {
                "question_id": question["question_id"],
                "model": model,
                "mode": mode,
                "rank": hit.rank,
                "chunk_id": hit.chunk_id,
                "score": float(hit.score),
                "query": question["question"],
                "textbook_id": chunk["textbook_id"],
                "provider": chunk["provider"],
                "source_locator": chunk["source_locator"],
            }
        )
    return rows

## 7. Evidence metrics


In [ ]:
def score_one(
    question: dict,
    hits: list[tuple[str, float]],
    model_key: str,
    mode: str,
    bm25_weight=None,
    dense_weight=None,
) -> dict:
    ids = [chunk_id for chunk_id, _ in hits]
    assert len(ids) == len(set(ids))
    paths = gold_paths[question["question_id"]]
    relevant = set().union(*paths)
    rank = {chunk_id: position for position, chunk_id in enumerate(ids, 1)}
    first_rank = min((rank[cid] for cid in relevant if cid in rank), default=None)
    complete_rank = min(
        (max((rank.get(cid, math.inf) for cid in path), default=math.inf) for path in paths),
        default=math.inf,
    )
    complete_rank = None if math.isinf(complete_rank) else int(complete_rank)
    source_ids = {by_chunk[cid]["textbook_id"] for cid in relevant}
    providers = {by_chunk[cid]["provider"] for cid in relevant}
    row = {
        "question_id": question["question_id"],
        "evaluation_split": WORKING_SPLIT_POLICY[question["split"]],
        "annotation_status": question["annotation_status"],
        "model": model_key,
        "mode": mode,
        "bm25_weight": bm25_weight,
        "dense_weight": dense_weight,
        "scope": "cross_textbook" if len(source_ids) > 1 else "single_source",
        "provider_scope": "cross_provider" if len(providers) > 1 else "single_provider",
        "gold_textbook": "+".join(sorted(source_ids)),
        "gold_provider": "+".join(sorted(providers)),
        "first_relevant_rank": first_rank,
        "complete_evidence_rank": complete_rank,
        "mrr": 0.0 if first_rank is None else 1 / first_rank,
        "complete_evidence_mrr": 0.0 if complete_rank is None else 1 / complete_rank,
        "returned_count": len(hits),
        "empty_retrieval": int(not hits),
        "ranked_chunk_ids": ids,
    }
    for k in TOP_K:
        top = set(ids[:k])
        row[f"hit_at_{k}"] = int(bool(top & relevant))
        row[f"complete_hit_at_{k}"] = int(any(path <= top for path in paths))
        row[f"evidence_recall_at_{k}"] = max(
            (len(path & top) / len(path) for path in paths),
            default=0.0,
        )
        row[f"core_precision_at_{k}"] = len(top & relevant) / len(top) if top else None
        row[f"no_relevant_hit_at_{k}"] = int(not (top & relevant))
    return row


def summarize(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    frame["run_name"] = frame["model"] + "_" + frame["mode"]
    for k in TOP_K:
        frame[f"context_noise_rate_at_{k}"] = 1 - frame[f"core_precision_at_{k}"]
        frame[f"returned_count_at_{k}"] = frame["returned_count"].clip(upper=k)
        frame[f"empty_retrieval_rate_at_{k}"] = frame[f"returned_count_at_{k}"].eq(0)
    metrics = ["mrr", "complete_evidence_mrr", "returned_count", "empty_retrieval"]
    metrics += [
        f"{metric}_at_{k}"
        for metric in (
            "hit",
            "complete_hit",
            "evidence_recall",
            "core_precision",
            "context_noise_rate",
            "no_relevant_hit",
            "returned_count",
            "empty_retrieval_rate",
        )
        for k in TOP_K
    ]
    groups = ["run_name", "model", "mode", "dense_weight", "bm25_weight"]
    grouped = frame.groupby(groups, dropna=False)
    return (
        grouped.size()
        .rename("question_count")
        .to_frame()
        .join(grouped[metrics].mean())
        .reset_index()
    )


def sort_summary(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.sort_values(
        ["complete_hit_at_5", "mrr", "run_name"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

## 8. Development sweep: 64 configurations


In [ ]:
def evaluate_dev() -> tuple[list[dict], list[dict], dict]:
    rows = []
    candidates = []
    provenance = {}

    baseline, _ = build_project_retriever("bge_m3", RetrievalMode.BM25)
    for question in dev_gold:
        result = baseline.retrieve(question["question"], FINAL_TOP_K)
        rows.append(score_one(question, hit_pairs(result), "bm25", "bm25"))
        candidates.extend(candidate_rows(question, "bm25", "bm25", result))
    del baseline

    for key in MODEL_SPECS:
        print("Dev:", key, flush=True)
        hybrid, artifact = build_project_retriever(key, RetrievalMode.HYBRID)
        provenance[key] = artifact
        for question in dev_gold:
            dense_result = hybrid.dense.retrieve(question["question"], FINAL_TOP_K)
            rows.append(score_one(question, hit_pairs(dense_result), key, "dense"))
            candidates.extend(candidate_rows(question, key, "dense", dense_result))
            for bm25_weight, dense_weight in HYBRID_WEIGHTS:
                hybrid.bm25_weight = bm25_weight
                hybrid.dense_weight = dense_weight
                mode = f"hybrid_d{dense_weight:.2f}_b{bm25_weight:.2f}"
                result = hybrid.retrieve(question["question"], FINAL_TOP_K)
                rows.append(
                    score_one(
                        question,
                        hit_pairs(result),
                        key,
                        mode,
                        bm25_weight,
                        dense_weight,
                    )
                )
                candidates.extend(candidate_rows(question, key, mode, result))
        del hybrid
        gc.collect()
    return rows, candidates, provenance


if RUN_EXPENSIVE:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    dev_rows, dev_candidates, index_provenance = evaluate_dev()
    assert len(dev_rows) == len(dev_gold) * 64
    dev_summary = sort_summary(summarize(dev_rows))
    dev_summary.to_csv(RESULT_DIR / "dev_summary.csv", index=False)
    display(dev_summary)
else:
    dev_rows = dev_candidates = []
    index_provenance = {}
    print("Dev sweep skipped. Set CS30_M6_RUN_MODELS=1 to run it.")

## 9. Dev baseline decision

The best Hybrid run is compared with BM25 question by question on complete
Hit@5. BM25 is replaced only if the candidate has a higher complete Hit@5,
does not reduce MRR, and the exact two-sided McNemar test gives `p < 0.05`.


In [ ]:
def exact_mcnemar_p(baseline_only: int, candidate_only: int) -> float:
    discordant = baseline_only + candidate_only
    if discordant == 0:
        return 1.0
    tail = sum(
        math.comb(discordant, successes)
        for successes in range(min(baseline_only, candidate_only) + 1)
    ) / (2**discordant)
    return min(1.0, 2 * tail)


if RUN_EXPENSIVE:
    best_hybrid = dev_summary.loc[
        dev_summary["mode"].str.startswith("hybrid_")
    ].iloc[0]
    candidate_model = best_hybrid["model"]
    candidate_mode = best_hybrid["mode"]
    baseline_by_question = {
        row["question_id"]: row
        for row in dev_rows
        if row["model"] == "bm25" and row["mode"] == "bm25"
    }
    candidate_by_question = {
        row["question_id"]: row
        for row in dev_rows
        if row["model"] == candidate_model and row["mode"] == candidate_mode
    }
    assert baseline_by_question.keys() == candidate_by_question.keys()

    baseline_only = sum(
        baseline_by_question[question_id]["complete_hit_at_5"]
        > candidate_by_question[question_id]["complete_hit_at_5"]
        for question_id in baseline_by_question
    )
    candidate_only = sum(
        candidate_by_question[question_id]["complete_hit_at_5"]
        > baseline_by_question[question_id]["complete_hit_at_5"]
        for question_id in baseline_by_question
    )
    mcnemar_p = exact_mcnemar_p(baseline_only, candidate_only)
    baseline_summary = dev_summary.loc[dev_summary["run_name"] == "bm25_bm25"].iloc[0]
    replace_baseline = (
        best_hybrid["complete_hit_at_5"] > baseline_summary["complete_hit_at_5"]
        and best_hybrid["mrr"] >= baseline_summary["mrr"]
        and mcnemar_p < 0.05
    )
    selection_decision = "replace_bm25" if replace_baseline else "retain_bm25"
    selection_reason = (
        "candidate passed the preregistered Dev replacement rule"
        if replace_baseline
        else "candidate did not pass the preregistered Dev replacement rule"
    )
    baseline_comparison = pd.DataFrame(
        [
            {
                "baseline_run_name": baseline_summary["run_name"],
                "candidate_run_name": best_hybrid["run_name"],
                "question_count": len(baseline_by_question),
                "baseline_complete_hits_at_5": int(
                    sum(row["complete_hit_at_5"] for row in baseline_by_question.values())
                ),
                "candidate_complete_hits_at_5": int(
                    sum(row["complete_hit_at_5"] for row in candidate_by_question.values())
                ),
                "baseline_mrr": float(baseline_summary["mrr"]),
                "candidate_mrr": float(best_hybrid["mrr"]),
                "baseline_only_hits": baseline_only,
                "candidate_only_hits": candidate_only,
                "exact_mcnemar_p": mcnemar_p,
                "selection_decision": selection_decision,
                "selection_reason": selection_reason,
            }
        ]
    )
    baseline_comparison.to_csv(
        RESULT_DIR / "dev_baseline_comparison.csv", index=False
    )
    display(baseline_comparison)
else:
    print("Dev baseline decision requires CS30_M6_RUN_MODELS=1.")


## 10. Diagnostic Test comparison

The 26-question Test split was previously evaluated across all 64
configurations. It is therefore already exposed and cannot support a pristine
one-time selection claim. This section reruns BM25 and the requested BGE-M3
Hybrid configuration on the same questions only as a transparent diagnostic
comparison. No model or weight is selected from these results.


In [ ]:
def evaluate_test_diagnostic() -> tuple[list[dict], list[dict]]:
    rows = []
    candidates = []
    hybrid, _ = build_project_retriever(
        DIAGNOSTIC_BGE_M3["model"],
        RetrievalMode.HYBRID,
        dense_weight=DIAGNOSTIC_BGE_M3["dense_weight"],
        bm25_weight=DIAGNOSTIC_BGE_M3["bm25_weight"],
    )
    for question in test_gold:
        baseline_result = hybrid.bm25.retrieve(question["question"], FINAL_TOP_K)
        rows.append(score_one(question, hit_pairs(baseline_result), "bm25", "bm25"))
        candidates.extend(candidate_rows(question, "bm25", "bm25", baseline_result))

        hybrid_result = hybrid.retrieve(question["question"], FINAL_TOP_K)
        rows.append(
            score_one(
                question,
                hit_pairs(hybrid_result),
                DIAGNOSTIC_BGE_M3["model"],
                DIAGNOSTIC_BGE_M3["mode"],
                DIAGNOSTIC_BGE_M3["bm25_weight"],
                DIAGNOSTIC_BGE_M3["dense_weight"],
            )
        )
        candidates.extend(
            candidate_rows(
                question,
                DIAGNOSTIC_BGE_M3["model"],
                DIAGNOSTIC_BGE_M3["mode"],
                hybrid_result,
            )
        )
    return rows, candidates


if RUN_EXPENSIVE:
    test_rows, test_candidates = evaluate_test_diagnostic()
    assert len(test_rows) == len(test_gold) * 2
    assert {row["model"] for row in test_rows} == {"bm25", "bge_m3"}
    test_summary = sort_summary(summarize(test_rows))
    assert len(test_summary) == 2
    test_summary.to_csv(RESULT_DIR / "test_diagnostic_summary.csv", index=False)
    display(test_summary)
else:
    test_rows = test_candidates = []
    print("Diagnostic Test comparison skipped. Set CS30_M6_RUN_MODELS=1 to run it.")


## 11. Save manifest and reviewable outputs

In [ ]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        "".join(
            json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n"
            for row in rows
        ),
        encoding="utf-8",
    )


if RUN_EXPENSIVE:
    write_jsonl(WORK_DIR / "gold_span_bindings.jsonl", binding_rows)
    write_jsonl(WORK_DIR / "dev_metrics.jsonl", dev_rows)
    write_jsonl(WORK_DIR / "dev_candidate_hits.jsonl", dev_candidates)
    write_jsonl(WORK_DIR / "test_diagnostic_metrics.jsonl", test_rows)
    write_jsonl(WORK_DIR / "test_diagnostic_candidate_hits.jsonl", test_candidates)

    git_state = capture_git_state(PROJECT_ROOT)
    result_files = [
        RESULT_DIR / "dev_summary.csv",
        RESULT_DIR / "dev_baseline_comparison.csv",
        RESULT_DIR / "test_diagnostic_summary.csv",
        RESULT_DIR / "truncation_diagnostics.csv",
    ]
    baseline_selection = json.loads(
        baseline_comparison.to_json(orient="records")
    )[0]
    manifest = {
        "schema_version": "m6-w6-v2-project-retriever-2",
        "created_at_utc": datetime.now(UTC).isoformat(),
        "reportable": False,
        "evaluation_set_version": "m6-v2-provisional-3book-dev162-test26-2026-10-13",
        "split_policy": "provisional_three_book_split_from_source_labels",
        "formal_target_split": {"dev": 60, "test": 180, "textbooks": 6},
        "dev_question_count": len(dev_gold),
        "test_question_count": len(test_gold),
        "test_configuration_count": 2,
        "test_policy": "diagnostic BM25 versus BGE-M3 on previously exposed Test",
        "test_exposure_status": "previously_exposed_to_64_configurations",
        "test_selection_used": False,
        "diagnostic_configuration": DIAGNOSTIC_BGE_M3,
        "baseline_selection": baseline_selection,
        "project_retriever": "cs30.pipeline.build_real_retrieval_deps",
        "rrf_input_top_k": RRF_INPUT_TOP_K,
        "rrf_k": RRF_K,
        "bm25_stopwords": True,
        "git": {
            "commit": git_state.commit,
            "dirty": git_state.dirty,
            "snapshot_sha256": git_state.snapshot_sha256,
        },
        "corpus_version": corpus_manifest["corpus_version"],
        "corpus_hash": corpus_manifest["corpus_hash"],
        "manifest_hash": corpus_manifest["manifest_hash"],
        "chunk_config_hash": corpus_manifest["chunk_config_hash"],
        "gold_sha256": sha256(gold_bytes),
        "binding_sha256": sha256(binding_bytes),
        "models": {
            key: {
                "model": spec["model"],
                "revision": spec["revision"],
                "trust_remote_code": spec["trust_remote_code"],
                "query_instruction": spec["query_instruction"],
                "query_prompt_name": spec["query_prompt_name"],
            }
            for key, spec in MODEL_SPECS.items()
        },
        "result_file_sha256": {
            path.name: sha256(path.read_bytes()) for path in result_files
        },
        "limitations": [
            "provisional 162/26 split rather than the formal six-textbook 60/180 split",
            "three OpenStax textbooks",
            "the 26-question Test set was previously exposed to all 64 configurations",
            "Gold annotation status is draft",
            "all current questions are answerable",
        ],
    }
    (RESULT_DIR / "run_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Saved reviewable outputs:", RESULT_DIR)
else:
    print("No measured files written during preflight-only execution.")


## 12. CSV-generated result tables

In [ ]:
def display_csv(name: str) -> None:
    path = RESULT_DIR / name
    if not path.is_file():
        print("Pending measured output:", path.name)
        return
    frame = pd.read_csv(path)
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(frame)


display_csv("dev_summary.csv")
display_csv("dev_baseline_comparison.csv")
display_csv("test_diagnostic_summary.csv")
display_csv("truncation_diagnostics.csv")


## Interpretation guardrails

- This is a provisional three-textbook development evaluation, not the formal
  six-textbook W6 gate.
- The current Gold rows are answerable-only and retain their source annotation
  status.
- The Dev replacement rule retains BM25: the best Hybrid gains two complete
  Hit@5 questions, has lower MRR, and is not significant by the exact paired
  McNemar test.
- The 26-question Test set was already exposed to all 64 configurations. The
  BM25 and BGE-M3 numbers shown here are diagnostic only and must not be used to
  retune or claim an unbiased final comparison.
- The next PR may add oralized-query, cross-encoder reranking, lexical-overlap
  bias, and refusal-threshold experiments only after their hypotheses are
  registered in the ablation plan.
